# Research 65 — RealMLP-TD default OOF (GPU)

**Hypothesis:** RealMLP, a tabular MLP trained from scratch by pytabkit with its tuned-default (TD) settings, may add diversity to the reference. One fixed configuration, no tuning. It needs numeric columns without NaN, so median imputation is fitted on each fold's training rows only. Same three stratified folds (seed 42) as the reference. Predeclared decision: promote only if the fixed 80/20 blend beats the reference on macro F1 and is not worse on weighted F1. Train OOF only; Test is not loaded.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pytabkit import RealMLP_TD_Classifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)
ref = pd.read_csv(ROOT / "predictions/segment-oof.csv")
assert len(ref) == len(y) and np.array_equal(ref.segment.to_numpy(), y)
p_ref = ref[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
p_ref = p_ref / p_ref.sum(axis=1, keepdims=True)
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(train, y))

C:\Users\Gunte\Workspace\Workspace\works\CPE\ml00x\task2\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


W1109 20:03:02.754000 96448 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


In [2]:
def features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (a, b) in ratios.items():
        x[name] = x[a] / (x[b].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


X = features(train)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print("features", X.shape[1], "categorical", len(cat_cols), "numeric", len(num_cols))

features 51 categorical 10 numeric 41


In [3]:
p_cand = np.zeros((len(y), 4), dtype=float)
started = time.perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(folds, start=1):
    x_fit = X.iloc[fit_idx].copy()
    x_valid = X.iloc[valid_idx].copy()
    x_fit[cat_cols] = x_fit[cat_cols].fillna("__MISSING__").astype(str)
    x_valid[cat_cols] = x_valid[cat_cols].fillna("__MISSING__").astype(str)
    medians = x_fit[num_cols].median()
    x_fit[num_cols] = x_fit[num_cols].fillna(medians)
    x_valid[num_cols] = x_valid[num_cols].fillna(medians)
    model = RealMLP_TD_Classifier(device="cuda", random_state=42, n_threads=4, verbosity=0)
    model.fit(x_fit, y[fit_idx], cat_col_names=cat_cols)
    p_cand[valid_idx] = model.predict_proba(x_valid)
    fold_f1 = f1_score(y[valid_idx], p_cand[valid_idx].argmax(1), average="macro")
    print(f"Fold {fold}: RealMLP-TD macro F1={fold_f1:.6f}", flush=True)
print(f"Elapsed seconds={time.perf_counter() - started:.1f}. Train OOF only; Test not loaded.")

C:\Users\Gunte\Workspace\Workspace\works\CPE\ml00x\task2\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


C:\Users\Gunte\Workspace\Workspace\works\CPE\ml00x\task2\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Fold 1: RealMLP-TD macro F1=0.752660


C:\Users\Gunte\Workspace\Workspace\works\CPE\ml00x\task2\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


C:\Users\Gunte\Workspace\Workspace\works\CPE\ml00x\task2\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Fold 2: RealMLP-TD macro F1=0.753585


C:\Users\Gunte\Workspace\Workspace\works\CPE\ml00x\task2\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


C:\Users\Gunte\Workspace\Workspace\works\CPE\ml00x\task2\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Fold 3: RealMLP-TD macro F1=0.756429


Elapsed seconds=3106.7. Train OOF only; Test not loaded.


In [4]:
def score_table(name, p_cand):
    ref_folds = [f1_score(y[v], p_ref[v].argmax(1), average="macro") for _, v in folds]
    rows = []
    for label, probs in [
        ("Reference equal CatBoost/LightGBM", p_ref),
        (f"{name} alone", p_cand),
        (f"Fixed 80/20 reference + {name}", 0.8 * p_ref + 0.2 * p_cand),
    ]:
        pred = probs.argmax(axis=1)
        row = {
            "candidate": label,
            "macro_f1": f1_score(y, pred, average="macro"),
            "weighted_f1": f1_score(y, pred, average="weighted"),
        }
        for k, (_, v) in enumerate(folds, start=1):
            row[f"fold_{k}_delta_vs_reference"] = (
                f1_score(y[v], pred[v], average="macro") - ref_folds[k - 1]
            )
        rows.append(row)
    return pd.DataFrame(rows)

In [5]:
results = score_table("RealMLP-TD", p_cand)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-realmlp-td-results.csv", index=False)
pd.DataFrame(p_cand, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-realmlp-td-oof.csv", index=False
)

,candidate,macro_f1,weighted_f1,fold_1_delta_vs_reference,fold_2_delta_vs_reference,fold_3_delta_vs_reference
0,Reference equal CatBoost/LightGBM,0.750895,0.764683,0.000000,0.000000,0.000000
1,RealMLP-TD alone,0.754231,0.767629,0.003215,0.003765,0.003039
2,Fixed 80/20 reference + RealMLP-TD,0.752264,0.765910,0.000901,0.001052,0.002162


## Research decision

Decide on the fixed 80/20 row above using the predeclared rule. This is one configuration; no hyperparameter search was run.